In [1]:
import pickle
import boto3
import botocore
from botocore.exceptions import ClientError
import os, time, json, time
from datetime import datetime

from misc import load_from_yaml, save_to_yaml
import s3, iam, lftn, glue, lambdafn as lfn, sns, eventbridge as event, rds, networking
from lambdafn import build_lambda_package, print_latest_lambda_logs

from redshift_deploy import create_development_cluster
from redshift_manager import RedshiftClusterConfig, NetworkConfig, SecurityConfig, RedshiftClusterManager

from dotenv import load_dotenv

load_dotenv(os.getenv("MY_AWS_DIR", "") + "/.env")

from mylogger import CustomLogger

logger = CustomLogger()

In [2]:
ACCOUNT_ID = os.environ["AWS_ACCOUNT_ID_ROOT"]
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

logger.info(f"VPC_ID: {ACCOUNT_ID}")

INFO: 2025-08-03 23:10:35 [1682100376.py:4] VPC_ID: 530976901147


In [3]:
rds_client           = boto3.client('rds', region_name=REGION)
iam_client           = boto3.client('iam', region_name=REGION)
s3_client            = boto3.client('s3', region_name=REGION)
glue_client          = boto3.client('glue', region_name=REGION)
lakeformation_client = boto3.client("lakeformation", region_name=REGION)
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
events_client        = boto3.client('events', region_name=REGION)
lambda_client        = boto3.client('lambda', region_name=REGION)

# Create a CloudWatch client for Logs
logs_client = boto3.client("logs", region_name=REGION)

redshift_client = boto3.client("redshift", region_name=REGION)

In [4]:
S3_BUCKET_DATALAKE = "htech-datalake-bkt"
S3_BUCKET_GLUE_ASSETS = "htech-glue-assets-bkt"
TEM_DIR = f"s3://{S3_BUCKET_GLUE_ASSETS}/temporary/"
SPARK_EVENT_LOG_PATH = f"s3://{S3_BUCKET_GLUE_ASSETS}/sparkHistoryLogs/"

GLUE_CATALOG_DB = "htech-glue-catalog-db"
DATALAKE_LOCATION_URI = f"s3://{S3_BUCKET_DATALAKE}"

GLUE_ROLE_NAME = "glue-pipeline-role"
LFN_ROLE_NAME = "lfn-pipeline-role"
GLUE_ROLE_ARN = "arn:aws:iam::530976901147:role/glue-pipeline-role"
LFN_ROLE_ARN = "arn:aws:iam::530976901147:role/lfn-pipeline-role"

#### Launch Redshift Cluster

In [5]:
RS_CLUSTER_IDENTIFIER="dev-rs-cluster"
RS_MASTER_USERNAME=os.environ["USERNAME"]
RS_MASTER_PASSWORD=os.environ["PASSWORD"]
RS_DATABASE_NAME="dev-rs-db"

In [8]:
print(RS_MASTER_PASSWORD)

admin23646


In [6]:
redshift_client.delete_cluster_subnet_group(
    ClusterSubnetGroupName="dev-rs-subnet-group"
)

{'ResponseMetadata': {'RequestId': 'ed58976e-ff20-48b2-a356-9bcea329e8c6',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'ed58976e-ff20-48b2-a356-9bcea329e8c6',
   'content-type': 'text/xml',
   'content-length': '232',
   'date': 'Mon, 04 Aug 2025 04:10:39 GMT'},
  'RetryAttempts': 0}}

In [ ]:
cluster_config = RedshiftClusterConfig(
    cluster_identifier=RS_CLUSTER_IDENTIFIER,
    master_username=RS_MASTER_USERNAME,
    master_password=RS_MASTER_PASSWORD,
    node_type="ra3.xlplus",
    cluster_type="single-node",  # Single node for dev
    database_name=RS_DATABASE_NAME,
    publicly_accessible=False,
    encrypted=True,
    automated_snapshot_retention_period=1,  # Shorter retention for dev
    tags={
        "Environment": "development",
        "Team": "HTech",
        "CostCenter": "DE",
    },
)

# More restrictive network for dev
network_config = NetworkConfig(
    vpc_cidr="10.1.0.0/16", subnet_cidrs=["10.1.1.0/24", "10.1.2.0/24"]
)

# Restrictive security - only VPC access
security_config = SecurityConfig(allowed_cidr_blocks=["10.1.0.0/16"], port=5439)

manager = RedshiftClusterManager()

result = manager.create_complete_redshift_environment(
    cluster_config=cluster_config,
    network_config=network_config,
    security_config=security_config,
    resource_prefix="dev-rs",
    create_iam_role=True,
    wait_for_available=True,
)
print(result)

In [ ]:
RESOURCE_FILE = "/Users/am/mydocs/Software_Development/Web_Development/aws/aws_redshift/rs_cluster_resources.yaml"
manager.serialize_resources(RESOURCE_FILE)

In [ ]:
RS_RESOURCES = load_from_yaml(RESOURCE_FILE)
logger.info(RS_RESOURCES)

-   **ClusterSecurityGroups (list)**:

    -   A list of security groups to be associated with this cluster.
    -   Default: The default cluster security group for Amazon Redshift.

-   **VpcSecurityGroupIds (list)**:

    -   A list of Virtual Private Cloud (VPC) security groups to be associated with the cluster.
    -   Default: The default VPC security group is associated with the cluster.

-   **ClusterSubnetGroupName (string)**:

    -   The name of a cluster subnet group to be associated with this cluster.
    -   If this parameter is not provided the resulting cluster will be deployed outside virtual private cloud (VPC).

-   **AvailabilityZone (string)**:

    -   The EC2 Availability Zone (AZ) in which you want Amazon Redshift to provision the cluster. For example, if you have several EC2 instances running in a specific Availability Zone, then you might want the cluster to be provisioned in the same zone in order to decrease network latency.
    -   Default: A random, system-chosen Availability Zone in the region that is specified by the endpoint.
    -   Constraint: The specified Availability Zone must be in the same region as the current endpoint.


In [ ]:
# # Associate the IAM role with the Redshift cluster (OPTIONAL)
# response = redshift_client.modify_cluster_iam_roles(
#     ClusterIdentifier=cluster_identifier,
#     AddIamRoles=[power_user_access_policy_arn]  # This adds the role to the cluster
# )

### [AWS Tutorials: Using Amazon Redshift in AWS based Data Lake](https://www.youtube.com/watch?v=Co8UpEYlZYA&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=1&t=1078s)

-   Glue Job sourceing from Redshift Table.

In [ ]:
RS_ROLE_NAME = "dev-rs-role"
RS_CRAWLER_NAME = "dev-rs-crawler"
RS_CLUSTER_IDENTIFIER="dev-rs-cluster"
RS_MASTER_USERNAME=os.environ["USERNAME"]
RS_MASTER_PASSWORD=os.environ["PASSWORD"]
RS_DATABASE_NAME="dev-rs-db"
AWS_DEFAULT_ROUTE_TABLE =""

-   [lab](https://aws-dojo.com/ws30/labs/)

-   <details><summary style="font-size:20px;color:Orange"><a href="">screenshots</a></summary>

    <div style="text-align:center" ><img src="./images/screenshot.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 1.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 2.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 3.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 4.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 5.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 6.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 7.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 8.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 9.png" width="1000px" height="500px" /></div>
    <div style="text-align:center" ><img src="./images/screenshot 10.png" width="1000px" height="500px" /></div>

    </details>

#### Create IAM Role (for AWS Glue Service)

- Create aws glue role by the name of `glue_role_name`.
- Assign Power User Access Policy (`PowerUserAccess`) to the role.

In [ ]:
assume_role_policy_doc = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "glue.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
GLUE_ROLE_ARN = iam_client.create_role(
    RoleName=GLUE_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(assume_role_policy_doc),
    Description="Glue Service Role"
)['Role']['Arn']

In [ ]:
# aws_glue_service_policy_arn = "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"
# admin_access_policy_arn = "arn:aws:iam::aws:policy/AdministratorAccess"
power_user_access_policy_arn = "arn:aws:iam::aws:policy/PowerUserAccess"

In [ ]:
# Attach AWS managed policy with the role
response = iam_client.attach_role_policy(
    RoleName=GLUE_ROLE_NAME,
    PolicyArn=power_user_access_policy_arn
)

In [ ]:
assume_role_policy_doc = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "redshift.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
RS_ROLE_ARN = iam_client.create_role(
    RoleName=RS_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(assume_role_policy_doc),
    Description="Glue Service Role"
)['Role']['Arn']

In [ ]:
# aws_glue_service_policy_arn = "arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole"
# admin_access_policy_arn = "arn:aws:iam::aws:policy/AdministratorAccess"
amazon_redshift_all_commands_fullaccess = "arn:aws:iam::aws:policy/AmazonRedshiftAllCommandsFullAccess"
amazon_s3_read_only_access = "arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess"

In [ ]:
# Attach AWS managed policy with the role
response = iam_client.attach_role_policy(
    RoleName=RS_ROLE_NAME,
    PolicyArn=amazon_redshift_all_commands_fullaccess
)
response = iam_client.attach_role_policy(
    RoleName=RS_ROLE_NAME,
    PolicyArn=amazon_s3_read_only_access
)

#### Create S3 Bucket and Folders

In [ ]:
output,scripts,tmp = ['output', 'scripts', 'tmp']     # List of folders to create

s3.create_s3_bucket(S3_BUCKET_DATALAKE, [output, scripts, tmp])

#### Create Private Link (VPC Endpoint)

-   `Gateway` endpoints serve as a target for a route in your route table for traffic destined for the service.

In [ ]:
SERVICE_NAME = 'com.amazonaws.us-east-1.s3'

# Create an VPC (Gateway Interface) Endpoint
vpc_endpoint_id = ec2_client.create_vpc_endpoint(
    VpcEndpointType='Gateway',
    VpcId=RS_RESOURCES["vpc_id"],
    ServiceName=SERVICE_NAME,
    RouteTableIds=[RS_RESOURCES["route_table_id"]],
    PrivateDnsEnabled=False  # Enable private DNS to resolve service names within the VPC
)['VpcEndpoint']['VpcEndpointId']

In [ ]:
# ec2_client.create_tags(Resources=['vpc_endpoint_id'],Tags=[{'Key': 'Name', 'Value': 'rs-glue-vpc-endpoint'}])

#### Create glue components

##### Create Glue Catalog Database 

In [ ]:
## Example usage
datalake_location_uri = f"s3://{S3_BUCKET_DATALAKE}" #/{datalake_folder_name}"

create_database_response = glue_client.create_database(
    CatalogId=ACCOUNT_ID,
    DatabaseInput={
        'Name': GLUE_CATALOG_DB,
        'Description': 'This is a Glue Catalog database',
        'LocationUri': datalake_location_uri,
    }
)
logger.info(create_database_response)

- Grant `CREATE_TABLE` permission to `glue_role_name` on data catalog DB.

In [ ]:
# Grant `glue_role_name` Role the 'CREATE_TABLE' LF Permission
response = lakeformation_client.grant_permissions(
    Principal={
        'DataLakePrincipalIdentifier': GLUE_ROLE_ARN
    },
    Resource={
        'Database': {
            'Name': GLUE_CATALOG_DB
        }
    },
    Permissions=['CREATE_TABLE', 'ALTER'],
    PermissionsWithGrantOption=[]
)

In [ ]:
glue_client.delete_database(CatalogId=ACCOUNT_ID,Name=GLUE_CATALOG_DB)

##### Create JDBC connection for Glue Crawler.

-   <b style="color:red">InvalidInputException</b>: At least one security group must open all ingress ports.To limit traffic, the source security group in your inbound rule can be restricted to the same security group
- **One of the security groups that's associated with the connection must have a self-referenced inbound rule that's open to all TCP ports. One of the security groups must be open to all outbound traffic.**
    ```python
    ingress_rules.append(
        {
            "IpProtocol": "tcp",
            "FromPort": 0,
            "ToPort": 65535,
            "UserIdGroupPairs": [
                {
                    "GroupId": this_security_group_id,  # Same SG it's attached to
                }
            ],
        }
    )
    ```

In [ ]:
# glue_client.delete_connection?
# glue_client.get_connection?

In [ ]:
# ?glue_client.create_connection

In [ ]:
glue_rs_connection_name = "glue-rs-connection"
port = '5439'
host_name = 'dev-rs-cluster.cxoevethw4s6.us-east-1.redshift.amazonaws.com'

# Construct the connection properties
jdbc_url = f"jdbc:redshift://{host_name}:{port}/{RS_DATABASE_NAME}"
connection_properties = {
    "USERNAME": RS_MASTER_USERNAME,
    "PASSWORD": RS_MASTER_PASSWORD,
    "JDBC_CONNECTION_URL": jdbc_url,
    "JDBC_ENFORCE_SSL": "false",  # set to 'true' if using SSL
}

# Construct the physical connection requirements
physical_connection_requirements = {
    "SecurityGroupIdList": [RS_RESOURCES["security_group_id"]],
    "SubnetId": RS_RESOURCES["subnet_ids"][0],  # Use subnet ID instead of VPC ID
}

response = glue_client.create_connection(
    ConnectionInput={
        "Name": glue_rs_connection_name,
        "ConnectionType": "JDBC",  # Use JDBC for Redshift
        "ConnectionProperties": connection_properties,
        "PhysicalConnectionRequirements": physical_connection_requirements
    },
    Tags={'Name': f"{glue_rs_connection_name}"}
)

In [ ]:
# logger.info(glue_client.get_connection(Name=glue_rs_connection_name))

- Test the Connection:
    -   <b style="color:red">FAILED</b>: For some unknown reasons connection made Using the SDK (Boto3) does not work unless you make some random eidt on the connection from AWS Console.

In [ ]:
# glue_mysql_connection_name = "glue-mysql-connection"
# response = glue_client.get_connection(Name=glue_rh_connection_name)
# print(response)

In [ ]:
# glue_client.delete_connection(ConnectionName=glue_rh_connection_name)

##### Create Glue Crawler.

In [ ]:
create_crawler_response1 = glue_client.create_crawler(
    Name=RS_CRAWLER_NAME,
    Role=GLUE_ROLE_ARN,
    DatabaseName=GLUE_CATALOG_DB,
    Description="Crawler for generated customer schema",
    Targets={
        "JdbcTargets": [
            {
                "ConnectionName": glue_rs_connection_name,
                "Path": f"{RS_DATABASE_NAME}/%",
                "Exclusions": [],  # Optional: specify any patterns to exclude
            }
        ],
    },
    SchemaChangePolicy={
        "UpdateBehavior": "UPDATE_IN_DATABASE",
        "DeleteBehavior": "DELETE_FROM_DATABASE",
    },
    RecrawlPolicy={"RecrawlBehavior": "CRAWL_EVERYTHING"},
)

In [ ]:
# logger.info(create_crawler_response1)

In [ ]:
# run_crawler_response1 = glue_client.start_crawler(Name=rds_crawler_name)
# print(run_crawler_response1)

In [ ]:
# ?lakeformation_client.grant_permissions

- Grant Table level LF permission (`SELECT`) to `glue_role_name` on the tables just created on Catalog DB.

In [ ]:
response = lakeformation_client.grant_permissions(
    Principal={"DataLakePrincipalIdentifier": GLUE_ROLE_ARN},
    Resource={"Table": {"DatabaseName": f"{GLUE_CATALOG_DB}", "TableWildcard": {}}},
    Permissions=["SELECT"],
    PermissionsWithGrantOption=[],
)

In [ ]:
logger.info(glue_client.get_database(Name=GLUE_CATALOG_DB))

#### Delete All Resources

In [ ]:
# s3_resources = boto3.resource('s3')
# bucket = s3_resources.Bucket(bucket_name)

# # Delete all objects in the bucket
# bucket.objects.all().delete()

# # Delete all object versions (if versioning is enabled)
# bucket.object_versions.all().delete()

# # Finally, delete the bucket
# bucket.delete()


In [ ]:
# Delete the Redshift cluster without a final snapshot
response = redshift_client.delete_cluster(
    ClusterIdentifier=RS_RESOURCES["cluster_identifier"],
    SkipFinalClusterSnapshot=True,  # Set to False if you want to take a final snapshot before deletion
)
logger.info(response)

In [ ]:
# rds.delete_rds_instance(db_instance_identifier_mysqlrds1)

In [ ]:
response = glue_client.delete_connection(ConnectionName=glue_rs_connection_name)
response = glue_client.delete_crawler(Name=RS_CRAWLER_NAME)

In [ ]:
# iam.delete_iam_role(glue_role_name)

In [ ]:
# Delete the VPC Endpoint
response = ec2_client.delete_vpc_endpoints(
    VpcEndpointIds=[vpc_endpoint_id]
)

### [How to Load Data from S3 Bucket into Amazon Redshift using COPY command [Part4]](https://www.youtube.com/watch?v=5WhEeGvKn8w)

```sql
-- STEP-1: Create Schema in Redshift Database.
CREATE SCHEMA IF NOT EXISTS MY_SALES;

-- STEP-2: Create Table
CREATE TABLE MY_SALES.SALES(
    CUSTOMER_ID INT,
    NAME VARCHAR(100),
    SALES DECIMAL
);

-- STEP-3: Copy data from S3 data-location into Redshift tables.
COPY MY_SALES.SALES
FROM 's3://htech-datalake-bkt/raw/sales_tiny/'
IAM_ROLE 'arn:aws:iam::530976901147:role/dev-rs-role'
REGION 'us-east-1'
IGNOREHEADER 1
IGNOREBLANKLINES
DELIMITER ','
DATEFORMAT 'auto';

-- STEP-3: Test copy operations by selecting date from the Table.
SELECT * FROM MY_SALES.SALES;
```

### [AWS Tutorials - Access Glue Catalog using Amazon Redshift Spectrum](https://www.youtube.com/watch?v=UhBCpNmM2CE&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=2)

-   <details><summary style="font-size:20px;color:Orange"><a href="">screenshots</a></summary>

    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 1.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 2.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 3.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 4.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 5.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 6.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 7.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 8.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/Access-Glue-Catalog-using-Amazon-Redshift-Spectrum 9.png" width="900px" length="750px"/></div>

    </details>

To access the AWS Glue Catalog using Amazon Redshift Spectrum, you must first create an external schema in Redshift. This external schema acts as a link to a database in your Glue Data Catalog, allowing you to query external tables directly from Redshift without loading the data.

-----

- **Prerequisites**: Before you create the external schema, you'll need to set up a few things:

    * **IAM Role:** An IAM role with permissions for Redshift to access both the AWS Glue Data Catalog and the Amazon S3 bucket where your data files are stored. The role should have a trust policy allowing the Redshift service to assume it.
    * **S3 Data:** The data files you want to query must be in a supported format (like Parquet, ORC, or CSV) and stored in an S3 bucket.
    * **Glue Database and Tables:** The AWS Glue Data Catalog needs a database and tables defined. These table definitions, often created by an AWS Glue Crawler, should point to the data files in your S3 bucket.

-----


##### 1\. Create an IAM Role 🛡️

You need an IAM role that allows your Redshift cluster to perform actions on AWS Glue and S3. This role should have:

  * A trust relationship with `redshift.amazonaws.com`
  * Permissions to `s3:Get*` and `s3:List*` on your S3 bucket.
  * Permissions to `glue:Get*` on your Glue Data Catalog databases and tables.

##### 2\. Attach the IAM Role to Your Redshift Cluster 📎

After creating the role, you must associate it with your Redshift cluster. This gives the cluster the necessary permissions to access external resources.

##### 3\. Create the External Schema in Redshift 📝

Connect to your Redshift cluster and use the `CREATE EXTERNAL SCHEMA` SQL command. This is the key step that links your Redshift database to the Glue Data Catalog.

Here's the general syntax:

```sql
CREATE EXTERNAL SCHEMA <schema_name>
FROM DATA CATALOG
DATABASE '<glue_database_name>'
IAM_ROLE '<iam_role_arn>'
CREATE EXTERNAL DATABASE IF NOT EXISTS;
```

  * **`schema_name`**: The name you want to give the external schema in Redshift (e.g., `spectrum_schema`).
  * **`glue_database_name`**: The name of the database in your AWS Glue Data Catalog that contains the tables you want to access.
  * **`iam_role_arn`**: The Amazon Resource Name (ARN) of the IAM role you created in the previous steps.
  * **`CREATE EXTERNAL DATABASE IF NOT EXISTS`**: An optional clause that creates a corresponding database in your Glue Data Catalog if it doesn't already exist.

##### 4\. Querying the External Tables 🔍

Once the external schema is created, you can query the tables within it as if they were regular Redshift tables. The tables will be listed under the external schema you just created. For example:

```sql
SELECT * FROM <schema_name>.<glue_table_name> LIMIT 10;
```

This query will be processed by Redshift Spectrum, which will read the metadata from the Glue Catalog and then read the data directly from the underlying files in S3.


```sql
CREATE EXTERNAL SCHEMA "dev-rs-db"."myschema"
FROM DATA CATALOG
DATABASE "htech-glue-catalog-db"
IAM_ROLE "arn:aws:iam::530976901147:role/dev-rs-role";
CREATE EXTERNAL DATABASE IF NOT EXISTS;
``

### [AWS Tutorials - Continuous S3 data ingestion to Amazon Redshift (Copy Job)](https://www.youtube.com/watch?v=2reIpdRYscM&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=2)

To configure AWS Redshift AUTO COPY from S3, you first need to establish the necessary prerequisites, then create a COPY job with the `AUTO ON` parameter. This automates the process of loading new files that arrive in a specified S3 bucket into your Redshift table.

-----

##### Prerequisites for Auto-Copy 🛠️

Before setting up the auto-copy job, ensure you have the following in place:

  * **IAM Role:** You need an AWS Identity and Access Management (IAM) role that grants Redshift permission to read from your S3 bucket. This role should have policies like `AmazonS3ReadOnlyAccess` and be attached to your Redshift cluster or workgroup.
  * **S3 Bucket and Data:** Your S3 bucket must be configured to store the data files you want to load. It's a good practice to use a specific folder (prefix) for the files that will be auto-copied. Files should be appropriately sized (between 1MB and 1GB is often recommended) and compressed using formats like GZIP or ZSTD for optimal performance.
  * **Redshift Table:** A table must be created in your Redshift database with a schema that matches the data files in your S3 bucket.

-----

##### Creating the Auto-Copy Job 🚀

You create the auto-copy job using a modified `COPY` command in your Redshift query editor. The key to enabling automatic ingestion is the `JOB CREATE` and `AUTO ON` parameters.

Here is the general syntax for the command:

```sql
COPY <table-name>
FROM 's3://<s3-bucket-name>/<prefix>/'
IAM_ROLE 'arn:aws:iam::<your-account-id>:role/<your-iam-role>'
[optional-copy-parameters]
JOB CREATE <job-name>
AUTO ON;
```

**Explanation of the parameters:**

  * **`COPY <table-name>`**: Specifies the target Redshift table where the data will be loaded.
  * **`FROM 's3://<s3-bucket-name>/<prefix>/'`**: The path to the S3 bucket and folder where the data files are located.
  * **`IAM_ROLE '...'`**: The ARN (Amazon Resource Name) of the IAM role you created with S3 access.
  * **`[optional-copy-parameters]`**: You can include additional `COPY` command parameters here, such as `FORMAT AS CSV`, `GZIP`, or a specific delimiter.
  * **`JOB CREATE <job-name>`**: Creates a new auto-copy job and assigns it a unique name.
  * **`AUTO ON`**: This is the crucial parameter that enables the continuous monitoring of the specified S3 path. When a new file is detected, Redshift automatically initiates the `COPY` command to load it into the table.

Once you execute this command, Redshift sets up an S3 event integration. This integration monitors the S3 bucket for new file uploads and triggers the `COPY` job to load the data without any manual intervention.

### [AWS Tutorials - Using Amazon Redshift Data APIs for ETL](https://www.youtube.com/watch?v=4dnp8pm5dLA&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=7&t=55s)

-   <details><summary style="font-size:20px;color:Orange"><a href="https://www.youtube.com/watch?v=4dnp8pm5dLA&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=7&t=55s">screenshots</a></summary>

    <div style="text-align:center"><img src="./images/screenshot 12.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 13.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 14.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 15.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 16.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 17.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 18.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 19.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 20.png" width="900px" length="750px"/></div>
    <div style="text-align:center"><img src="./images/screenshot 21.png" width="900px" length="750px"/></div>

    </details>

In [ ]:
# Initialize AWS clients
redshift_client = boto3.client('redshift')
secrets_manager_client = boto3.client('secretsmanager')
iam_client = boto3.client('iam')
sagemaker_client = boto3.client('sagemaker')

def create_redshift_cluster():
    """
    Creates an Amazon Redshift cluster.
    """
    cluster_identifier = "dojoredshift"
    response = redshift_client.create_cluster(
        ClusterIdentifier=cluster_identifier,
        NodeType='dc2.large',
        MasterUsername='awsuser',
        MasterUserPassword='Password1!',
        ClusterType='single-node'
    )
    print(f"Creating Redshift cluster: {cluster_identifier}")
    return cluster_identifier

def wait_for_redshift(cluster_identifier):
    """
    Waits until the Redshift cluster is available.
    """
    print(f"Waiting for Redshift cluster '{cluster_identifier}' to become available...")
    while True:
        response = redshift_client.describe_clusters(ClusterIdentifier=cluster_identifier)
        cluster_status = response['Clusters'][0]['ClusterStatus']
        if cluster_status == 'available':
            print(f"Redshift cluster '{cluster_identifier}' is available.")
            break
        else:
            print(f"Cluster status: {cluster_status}")
            time.sleep(30)

def create_secret():
    """
    Creates a secret in AWS Secrets Manager for Redshift credentials.
    """
    secret_name = "dojosecret"
    secret_value = {
        "username": "awsuser",
        "password": "Password1!",
        "engine": "redshift",
        "host": "dojoredshift.cluster-identifier.aws-region.redshift.amazonaws.com",
        "port": 5439,
        "dbClusterIdentifier": "dojoredshift"
    }
    response = secrets_manager_client.create_secret(
        Name=secret_name,
        SecretString=str(secret_value)
    )
    secret_arn = response['ARN']
    print(f"Created secret: {secret_name}, ARN: {secret_arn}")
    return secret_arn

def create_iam_role():
    """
    Creates an IAM role for SageMaker with the necessary permissions.
    """
    role_name = "dojosagemakerrole"
    assume_role_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "sagemaker.amazonaws.com"},
                "Action": "sts:AssumeRole"
            }
        ]
    }
    response = iam_client.create_role(
        RoleName=role_name,
        AssumeRolePolicyDocument=str(assume_role_policy)
    )
    iam_client.attach_role_policy(
        RoleName=role_name,
        PolicyArn="arn:aws:iam::aws:policy/PowerUserAccess"
    )
    print(f"Created IAM role: {role_name}")
    return role_name

def create_sagemaker_notebook(role_name):
    """
    Creates a SageMaker notebook instance.
    """
    notebook_name = "dojodataapinotebook"
    response = sagemaker_client.create_notebook_instance(
        NotebookInstanceName=notebook_name,
        InstanceType='ml.t2.medium',
        RoleArn=f"arn:aws:iam::{boto3.client('sts').get_caller_identity()['Account']}:role/{role_name}"
    )
    print(f"Creating SageMaker notebook: {notebook_name}")
    return notebook_name

def main():
    # Step 2: Launch Redshift Cluster
    cluster_identifier = create_redshift_cluster()
    wait_for_redshift(cluster_identifier)
    
    # Step 4: Configure Secrets Manager
    secret_arn = create_secret()

    # Step 5: Configure IAM Role and SageMaker Notebook
    role_name = create_iam_role()
    create_sagemaker_notebook(role_name)


### [AWS Tutorials - Merge Operation in Amazon Redshift using AWS Glue ETL Job](https://www.youtube.com/watch?v=PNSeVQCft4M)

### [AWS Tutorials: Using Lambda UDF with Amazon Redshift](https://www.youtube.com/watch?v=HqpwL7et4eQ&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=4&t=119s)

-   [lab](https://aws-dojo.com/excercises/excercise31/)

- **Introduction**
  - AWS Lambda can now be used to create a user-defined function (UDF) in Amazon Redshift.
  - UDFs can be utilized in both the `SELECT` and `WHERE` clauses of SQL queries.
  - The business logic for the UDF is defined in a Lambda function, which allows the use of programming languages like Python or Node.js.
  
- **Steps to Create UDF with Lambda in Redshift**
  - **IAM Role Creation**
    - Create an IAM role for the Redshift cluster.
    - This IAM role should have permission to invoke the Lambda function.
    - Redshift cluster needs permission to invoke Lambda for executing the query.
  
  - **Defining the External Function**
    - Use the `CREATE EXTERNAL FUNCTION` command in Redshift.
    - In the UDF, the Lambda function is referenced to perform the business logic.
    - When the UDF is called in a query, it invokes the Lambda function to return the result.

- **Lambda Function Structure**
  - **Output Format**
    - The result returned by the Lambda function must be a dictionary with four fields:
      - `success`: Indicates if the Lambda function execution was successful (`true` or `false`).
      - `error_message`: Contains an error description if the Lambda function fails.
      - `result`: The actual result, which can be an array of records.
      - `number_of_records`: Optional but recommended to indicate how many records are being returned.
  
  - **Return Example**
    - Pass an array of arguments, multiply values, and append the result to an array.
    - Return a dictionary that includes:
      - `success` status.
      - `result`: Array of the calculated values.
      - `number_of_records`: Optional, but can be added.

- **Creating the Lambda Function**
  - **Lambda Creation**
    - Create the Lambda function like any other Lambda in AWS.
    - Ensure proper formatting when returning results as a dictionary.
  
  - **Returning the Output**
    - Populate the `success` field based on exception handling.
    - If `success` is `false`, include an `error_message`.
    - Append results in an array and return it.

- **Using the UDF in Queries**
  - Example SQL query using the UDF:
    - Call the UDF within the `SELECT` clause.
    - The UDF will invoke the Lambda function to compute the result.
  
- **Lambda Invocation Frequency**
  - **Invocation Examples**
    - Example 1: Lambda is invoked for each row if the parameters are dynamic (e.g., columns in the table).
    - Example 2: Lambda is invoked only once if parameters are fixed values.
  
  - **Invocation in SELECT Clause**
    - If used in the `SELECT` clause, Lambda function processes multiple rows in one call, looping through arguments.
    - Ensure that Lambda execution does not exceed time limits when handling multiple rows.

- **Considerations**
  - Be cautious of Lambda's concurrency limits and costs.
  - Use Lambda functions efficiently to avoid unnecessary invocations.
  - Test queries to optimize Lambda usage and minimize invocation count.

- **Practical Example: Creating a UDF with Lambda**
  - Create a Redshift cluster and associate it with an IAM role.
  - Create tables and insert data for query testing.
  - Create a Lambda function in Python 3.8 to process quantity and price.
  - Define a UDF in Redshift that uses this Lambda function.
  - Query the data using the UDF, passing parameters like `quantity_order` and `price_each`.

### [AWS Tutorials: Amazon Redshift Federated Query with RDS PostgreSQL](https://www.youtube.com/watch?v=vJXZwkch2WY&list=PLO95rE9ahzRuUGYApNciILstNNJlvuc6g&index=3)
-   [lab](https://aws-dojo.com/ws37/labs/#google_vignette)